# Add standard sequence keys to the chirp (Euler) vec

The chirp stimulus is **one sequence repeated 20 times** (no directions). This notebook
adds a *sequence key* to the vec's last column so the standard vec analysis
(`utils.group_triggers_by_sequence` / `build_spikes_per_sequence_dict`) can split spikes
per repetition, the same way it does for the drifting gratings.

Key format (see Standard_Vec_Analysis): `<sequence-type><repetition>` with the last
`n_digit_for_rep` (=4) digits being the repetition. Here the sequence type is always `1`,
so repetition `r` gets key `10000 + r` (`10000` .. `10019`).

**Repetition boundaries.** The chirp waveform (vec column 1) repeats every 1600 frames
(32 s at 50 Hz; verified below by autocorrelation), and the established pipeline aligns
repetition 0 at frame 151 (the original code plots one repetition as `euler_vec[151:1751, 1]`).
So repetition `i` covers frames `[151 + i*1600 : 151 + (i+1)*1600]`; the lead-in and any
trailing frames get key `0` (ignored by the analysis).

In [ ]:
import numpy as np

vec_file = "Euler_50Hz_20reps_1024x768pix.vec"

vec_header = np.loadtxt(vec_file, max_rows=1)
vec = np.loadtxt(vec_file)[1:, :]  # drop the header line (not a trigger)
print(f"Selected vec file : {vec_file}")
print(f"Vec file length   : {vec.shape[0]} frames")

In [ ]:
# Verify the repetition period from the luminance waveform (column 1).
waveform = vec[:, 1]

def mismatch(period):
    return np.mean((waveform[:-period] - waveform[period:]) ** 2)

candidates = range(1500, 1700)
best_period = min(candidates, key=mismatch)
print(f"Best repetition period : {best_period} frames ({best_period / 50:.0f} s at 50 Hz)")

In [ ]:
# ---- Constants (the chirp's known structure) ----
LEAD_IN_FRAMES = 151    # frames before repetition 0 (matches euler_vec[151:1751] in the original code)
FRAMES_PER_REP = 1600   # one repetition = 32 s at 50 Hz (verified above)
N_REPETITIONS = 20
SEQUENCE_TYPE = 1       # a single sequence type for the whole chirp
REP_KEY_PLACES = 10000  # 1 sequence-type digit + 4 repetition digits

# ---- Write the sequence key into the last column ----
vec_std = vec.copy()
vec_std[:, -1] = 0  # lead-in / trailing frames are ignored by the analysis

for rep in range(N_REPETITIONS):
    start = LEAD_IN_FRAMES + rep * FRAMES_PER_REP
    end = LEAD_IN_FRAMES + (rep + 1) * FRAMES_PER_REP
    vec_std[start:end, -1] = SEQUENCE_TYPE * REP_KEY_PLACES + rep

# Sanity check: 20 repetitions, each 1600 frames
keys = vec_std[:, -1].astype(int)
rep_keys, counts = np.unique(keys[keys != 0], return_counts=True)
print("repetition keys :", rep_keys)
print("frames per key  :", set(counts))

In [ ]:
full_std_vec = np.concatenate((vec_header.reshape(1, -1), vec_std), axis=0)

np.savetxt("Euler_50Hz_20reps_1024x768pix_std.vec", full_std_vec, fmt="%1.f")
print("Saved Euler_50Hz_20reps_1024x768pix_std.vec")